# Candidate-action head screen (`cas_hl`)

This notebook analyses the heterogeneous-line candidate-action scoring sweep. The design is a full factorial:

- **pooling:** `mean` pools all affected nodes together; `typed_mean` concatenates separate means for affected busbars, lines, loads, and generators;
- **action features:** `f0` excludes and `f1` includes the static candidate-action feature vector;
- **action-0 head:** `a0h0` scores action 0 with the shared candidate scorer; `a0h1` replaces its score with a dedicated state-dependent MLP;
- **seed:** `s0`, `s1`, and `s2` are declared, while the notebook uses whichever runs have actually been downloaded.

The training curves are ten-episode evaluations and are used to study learning dynamics. They are not treated as final test estimates. The later **full-test** section uses the deterministic 201-chronic evaluation of each selected best checkpoint and remains empty, without failing, until those JSON files are downloaded.

> The held-out split is used during training for checkpoint selection, so scientifically it is a validation split. A separate untouched split is still required for an unbiased final generalization result.

In [1]:
from pathlib import Path
import importlib
import json
import sys
import tomllib

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

for candidate in [Path.cwd(), *Path.cwd().parents]:
    helpers = candidate / "helpers"
    repo_helpers = candidate / "Topology_Task" / "analysis" / "metrics" / "helpers"
    if helpers.exists() and (helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(helpers))
        break
    if repo_helpers.exists() and (repo_helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(repo_helpers))
        break
else:
    raise FileNotFoundError("Could not locate Topology_Task/analysis/metrics/helpers")

import wandb_metrics as wm
wm = importlib.reload(wm)
import survival_comparison as sc
sc = importlib.reload(sc)

print("wandb_metrics:", wm.__file__)
print("survival_comparison:", sc.__file__)
print("task directory:", wm.TASK_DIR)

wandb_metrics: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/wandb_metrics.py
survival_comparison: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/survival_comparison.py
task directory: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task


## Analysis controls

The comparison budget defaults to the last evaluation step reached by every downloaded run. This prevents a shorter run from being compared with a longer run at different training budgets.

In [2]:
USE_LOCAL_CACHE_ONLY = True
SMOOTH_WINDOW = 5
FINAL_WINDOW_EVALS = 10
TARGET_BUDGET_STEPS = 15_000_000
COMPARISON_BUDGET_STEPS = None  # None -> common horizon of downloaded runs
EXPECTED_FULL_TEST_EPISODES = 201

CONFIG_DIR = wm.TASK_DIR / "configs" / "gnn_action_scoring" / "candidate_variants"
FULL_TEST_EVAL_DIR = wm.TASK_DIR / "outputs" / "full_test_eval"
EXPORT_DIR = wm.TASK_DIR / "outputs" / "cas_hl_candidate_action_summary"

ACTOR_PARAM_METRIC = "model/actor_params"
TEST_ACTION0_METRICS = [f"test/explain/frac_action_0_agent_{index}" for index in range(3)]
EXTRA_METRICS = [ACTOR_PARAM_METRIC, *TEST_ACTION0_METRICS]

POOL_ORDER = ["mean", "typed_mean"]
POOL_LABELS = {"mean": "mean", "typed_mean": "typed mean"}
FEATURE_ORDER = [False, True]
ACTION0_ORDER = [False, True]

print("config folder:", CONFIG_DIR)
print("full-test search root:", FULL_TEST_EVAL_DIR)

config folder: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/gnn_action_scoring/candidate_variants
full-test search root: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/full_test_eval


## Reconstruct and validate the factorial design

Factors are read from TOML rather than trusted from the filename. The expected filename is reconstructed and checked, and settings that should be held constant are reported.

In [3]:
def config_record(path):
    with path.open("rb") as file:
        config = tomllib.load(file)
    args = config["args"]
    run_name = str(config.get("run", {}).get("name", path.stem))
    pool = str(args["candidate_action_pool"])
    use_features = bool(args["candidate_action_use_features"])
    use_action0_head = bool(args["candidate_action_do_nothing_head"])
    seed = int(args["seed"])
    pool_code = "tmean" if pool == "typed_mean" else pool
    feature_code = f"f{int(use_features)}"
    action0_code = f"a0h{int(use_action0_head)}"
    variant = f"{pool_code}_{feature_code}_{action0_code}"
    return {
        "config_path": str(path.relative_to(wm.TASK_DIR)),
        "run_name": run_name,
        "seed": seed,
        "pool": pool,
        "pool_label": POOL_LABELS.get(pool, pool),
        "pool_code": pool_code,
        "use_action_features": use_features,
        "feature_label": "features on" if use_features else "features off",
        "feature_code": feature_code,
        "use_action0_head": use_action0_head,
        "action0_label": "dedicated action-0" if use_action0_head else "shared action-0",
        "action0_code": action0_code,
        "variant": variant,
        "configured_steps": int(args["total_timesteps"]),
        "time_limit_minutes": int(args["time_limit"]),
        "cuda": bool(args["cuda"]),
        "actor_action_head": str(args["actor_action_head"]),
        "graph_type": str(args["gnn_graph_type"]),
        "gnn_type": str(args["gnn_type"]),
        "gnn_layers": int(args["gnn_layers"]),
        "gnn_hidden_dim": int(args["gnn_hidden_dim"]),
        "critic_encoder": str(args["critic_encoder"]),
        "chronic_split_seed": int(args["chronic_split_seed"]),
    }

config_paths = sorted(CONFIG_DIR.glob("cas_hl_*.toml"))
if not config_paths:
    raise FileNotFoundError(f"No cas_hl TOML files found under {CONFIG_DIR}")
config_catalog = pd.DataFrame(config_record(path) for path in config_paths)

expected_names = config_catalog.apply(
    lambda row: f"cas_hl_{row['variant']}_s{int(row['seed'])}", axis=1
)
mismatched = config_catalog[config_catalog["run_name"] != expected_names]
if not mismatched.empty:
    raise ValueError("Filename/config mismatch: " + ", ".join(mismatched["run_name"]))

print(f"Declared runs: {len(config_catalog)}")
for column in [
    "actor_action_head", "graph_type", "gnn_type", "gnn_layers",
    "gnn_hidden_dim", "critic_encoder", "chronic_split_seed",
    "configured_steps", "time_limit_minutes",
]:
    values = sorted(config_catalog[column].astype(str).unique())
    suffix = "" if len(values) == 1 else "  <-- NOT HELD CONSTANT"
    print(f"  {column}: {values}{suffix}")

design = config_catalog.pivot_table(
    index=["pool_code", "feature_code"],
    columns="action0_code",
    values="seed",
    aggfunc="nunique",
    fill_value=0,
)
display(design.rename_axis(index=["pool", "features"], columns="action-0 head"))

Declared runs: 24
  actor_action_head: ['candidate_pool']
  graph_type: ['heterogeneous_line']
  gnn_type: ['gine']
  gnn_layers: ['2']
  gnn_hidden_dim: ['128']
  critic_encoder: ['mlp']
  chronic_split_seed: ['0']
  configured_steps: ['15000000']
  time_limit_minutes: ['10080']


action-0 head   a0h0  a0h1
pool  features            
mean  f0           3     3
      f1           3     3
tmean f0           3     3
      f1           3     3

## Load the downloaded histories

Only exact run names declared by the sweep are selected. Missing seeds are reported and do not prevent the available data from being analysed.

In [4]:
for metric in EXTRA_METRICS:
    if metric not in wm.METRICS:
        wm.METRICS = [*wm.METRICS, metric]

requested_run_names = config_catalog["run_name"].tolist()
requested_run_name_set = set(requested_run_names)
wm.configure_run_filter_from_names(requested_run_names)
data = wm.load_wandb_data(use_local_cache_only=USE_LOCAL_CACHE_ONLY)
runs_df = data.runs_df.copy()
history_df = data.history_df.copy()

if not history_df.empty:
    history_df = history_df[history_df["run_name"].isin(requested_run_name_set)].copy()
if "name" in runs_df:
    runs_df = runs_df[runs_df["name"].isin(requested_run_name_set)].copy()
if history_df.empty:
    raise RuntimeError("No cas_hl histories were found in outputs/run_data.")

found_names = set(history_df["run_name"].unique())
print(f"Downloaded histories: {len(found_names)} / {len(requested_run_names)}")
missing_names = sorted(requested_run_name_set - found_names)
if missing_names:
    print(f"Missing ({len(missing_names)}):")
    for name in missing_names:
        print("  ", name)

Explicit run-name filter: 24 candidates
Project: corentin-plumet-epfl/Grid2Op
Task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
Cache mode: full
Cache dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache
Local-only mode: True
Force refresh: False
Refresh scan-history fallbacks: False
Selected 8 cached runs from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history
state
finished    8
History artifact setup: local_only=True, runs_df=8
[ 1/8] loading artifact cache: cas_hl_mean_f0_a0h0_s0
    loaded 192 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/cas_hl/runs/cas_hl_mean_f0_a0h0_s0__MAPPO_bus14_T_0_0__I__1785559080_3812/history.parquet in 0.1s
[ 2/8] loading artifact cache: cas_hl_mean_f0_a0h1_s0
    loaded 192 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/cas_hl/runs/cas_hl_mean_f0_a0h1_s0__MAPP

/Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/wandb_metrics.py:626: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat(pieces, ignore_index=True, sort=False)


(26880, 8)
Downloaded histories: 8 / 24
Missing (16):
   cas_hl_mean_f0_a0h0_s1
   cas_hl_mean_f0_a0h0_s2
   cas_hl_mean_f0_a0h1_s1
   cas_hl_mean_f0_a0h1_s2
   cas_hl_mean_f1_a0h0_s1
   cas_hl_mean_f1_a0h0_s2
   cas_hl_mean_f1_a0h1_s1
   cas_hl_mean_f1_a0h1_s2
   cas_hl_tmean_f0_a0h0_s1
   cas_hl_tmean_f0_a0h0_s2
   cas_hl_tmean_f0_a0h1_s1
   cas_hl_tmean_f0_a0h1_s2
   cas_hl_tmean_f1_a0h0_s1
   cas_hl_tmean_f1_a0h0_s2
   cas_hl_tmean_f1_a0h1_s1
   cas_hl_tmean_f1_a0h1_s2


## Coverage

A W&B run can be marked `finished` after stopping at its configured wall-time, so completion is assessed from the observed environment step rather than the W&B state.

In [5]:
observed_progress = (
    history_df.groupby("run_name", as_index=False)["step"]
    .max()
    .rename(columns={"step": "observed_steps"})
)
coverage = config_catalog.merge(observed_progress, on="run_name", how="left")
coverage["has_history"] = coverage["observed_steps"].notna()
coverage["observed_steps_m"] = coverage["observed_steps"] / 1_000_000
coverage["completion_pct"] = 100 * coverage["observed_steps"] / coverage["configured_steps"]
analysis_catalog = coverage[coverage["has_history"]].copy()

seed_counts = analysis_catalog.groupby("variant")["seed"].nunique()
max_seeds = int(seed_counts.max())
show_uncertainty = max_seeds > 1
uncertainty = "std" if show_uncertainty else None
print("Seeds present:", sorted(analysis_catalog["seed"].unique()))
print("Seed uncertainty:", "enabled" if show_uncertainty else "disabled (one seed per downloaded variant)")
if seed_counts.nunique() > 1:
    print("Uneven seed coverage:")
    display(seed_counts.rename("downloaded_seeds").reset_index())

display(coverage[[
    "variant", "seed", "run_name", "observed_steps_m",
    "completion_pct", "has_history",
]].sort_values(["variant", "seed"]).round(2))

coverage_figure = px.bar(
    analysis_catalog.sort_values("observed_steps_m"),
    x="observed_steps_m", y="run_name", color="pool_label",
    orientation="h", hover_data=["variant", "seed", "completion_pct"],
    title="cas_hl: downloaded training coverage",
    labels={"observed_steps_m": "Observed environment steps (millions)", "run_name": "Run"},
    height=max(480, 30 * len(analysis_catalog)),
)
coverage_figure.add_vline(x=TARGET_BUDGET_STEPS / 1_000_000, line_dash="dash", annotation_text="15M target")
coverage_figure.show()

Seeds present: [0]
Seed uncertainty: disabled (one seed per downloaded variant)


,variant,seed,run_name,observed_steps_m,completion_pct,has_history
0,mean_f0_a0h0,0,cas_hl_mean_f0_a0h0_s0,7.96,53.08,True
1,mean_f0_a0h0,1,cas_hl_mean_f0_a0h0_s1,NaN,NaN,False
2,mean_f0_a0h0,2,cas_hl_mean_f0_a0h0_s2,NaN,NaN,False
3,mean_f0_a0h1,0,cas_hl_mean_f0_a0h1_s0,7.96,53.08,True
4,mean_f0_a0h1,1,cas_hl_mean_f0_a0h1_s1,NaN,NaN,False
5,mean_f0_a0h1,2,cas_hl_mean_f0_a0h1_s2,NaN,NaN,False
6,mean_f1_a0h0,0,cas_hl_mean_f1_a0h0_s0,7.96,53.08,True
7,mean_f1_a0h0,1,cas_hl_mean_f1_a0h0_s1,NaN,NaN,False
8,mean_f1_a0h0,2,cas_hl_mean_f1_a0h0_s2,NaN,NaN,False
9,mean_f1_a0h1,0,cas_hl_mean_f1_a0h1_s0,7.96,53.08,True


## Episodic-survival curves

The displayed line is a trailing mean over `SMOOTH_WINDOW` evaluations. Raw logged values remain available in `survival_long`. With multiple downloaded seeds, curves are averaged per variant and the band shows one standard deviation across seeds.

In [6]:
survival_long = sc.extract_survival_curves(
    history_df, catalog=analysis_catalog, split="test", smooth=1
)
if survival_long.empty:
    raise RuntimeError("No test episodic-survival curves were found.")

max_curve_steps = survival_long.groupby("run_name")["step"].max()
automatic_common_budget = int(max_curve_steps.min())
comparison_budget = int(
    COMPARISON_BUDGET_STEPS if COMPARISON_BUDGET_STEPS is not None else automatic_common_budget
)
print("Metrics used:", sorted(survival_long["metric"].unique()))
print(f"Runs with test curves: {survival_long['run_name'].nunique()}")
print(f"Common comparison budget: {comparison_budget / 1_000_000:.3f}M steps")
if COMPARISON_BUDGET_STEPS is None:
    print("Set by shortest run:", max_curve_steps.idxmin())

variant_order = (
    analysis_catalog.sort_values(["pool", "use_action_features", "use_action0_head"])
    ["variant"].drop_duplicates().tolist()
)
palette = px.colors.qualitative.Dark24
variant_colors = {name: palette[index % len(palette)] for index, name in enumerate(variant_order)}

Metrics used: ['test/charts/episodic_survival']
Runs with test curves: 8
Common comparison budget: 7.963M steps
Set by shortest run: cas_hl_mean_f0_a0h0_s0


In [7]:
all_curves_figure = sc.plot_survival_comparison(
    survival_long,
    group_by="variant", label_by="variant",
    smooth=SMOOTH_WINDOW, uncertainty=uncertainty, min_members=1,
    show_members=show_uncertainty, colors=variant_colors,
    budget_step=comparison_budget,
    title="cas_hl: test episodic survival — all candidate-head variants",
    width=1450, height=720,
)
all_curves_figure.show()

### Controlled factor comparisons

Each panel changes one factor and holds the other two fixed. This is more interpretable than inferring a factor effect from the eight-curve overview.

In [8]:
factor_specs = [
    {
        "factor": "pool_label", "facet": "pooling_panel",
        "facet_values": lambda frame: frame["feature_code"] + " · " + frame["action0_code"],
        "title": "Pooling: mean versus typed mean",
        "colors": {"mean": "#1f77b4", "typed mean": "#d62728"},
    },
    {
        "factor": "feature_label", "facet": "feature_panel",
        "facet_values": lambda frame: frame["pool_code"] + " · " + frame["action0_code"],
        "title": "Static action features: off versus on",
        "colors": {"features off": "#7f7f7f", "features on": "#2ca02c"},
    },
    {
        "factor": "action0_label", "facet": "action0_panel",
        "facet_values": lambda frame: frame["pool_code"] + " · " + frame["feature_code"],
        "title": "Action 0: shared scorer versus dedicated head",
        "colors": {"shared action-0": "#9467bd", "dedicated action-0": "#ff7f0e"},
    },
]

for spec in factor_specs:
    facet_catalog = analysis_catalog.copy()
    facet_catalog[spec["facet"]] = spec["facet_values"](facet_catalog)
    facet_curves = survival_long.drop(columns=[spec["facet"]], errors="ignore").merge(
        facet_catalog[["run_name", spec["facet"]]], on="run_name", how="left"
    )
    figure = sc.plot_survival_comparison(
        facet_curves,
        group_by=spec["factor"], label_by=spec["factor"],
        facet_by=spec["facet"],
        smooth=SMOOTH_WINDOW, uncertainty=uncertainty, min_members=1,
        show_members=show_uncertainty, colors=spec["colors"],
        budget_step=comparison_budget, title=spec["title"],
        width=1450, height=850, ncols=2,
    )
    figure.show()

## Behavioural diagnostic: action-0 frequency

Because one screened factor changes how action 0 is scored, its deterministic evaluation frequency is plotted directly. The value is the mean fraction across the three agents at each evaluation.

In [9]:
action0_history = history_df[history_df["metric"].isin(TEST_ACTION0_METRICS)].copy()
if action0_history.empty:
    print("No test action-0 metrics were logged.")
else:
    action0_curves = (
        action0_history.groupby(["run_name", "step"], as_index=False)["value"].mean()
        .rename(columns={"value": "action0_fraction"})
        .merge(analysis_catalog, on="run_name", how="inner")
    )
    action0_curves["action0_pct"] = 100 * action0_curves["action0_fraction"]
    action0_figure = sc.plot_survival_comparison(
        action0_curves, group_by="variant", label_by="variant",
        value_col="action0_pct", smooth=SMOOTH_WINDOW, uncertainty=uncertainty,
        show_members=show_uncertainty, colors=variant_colors,
        budget_step=comparison_budget, y_range=[0, 100],
        y_title="Action-0 frequency across agents (%)",
        title="cas_hl: deterministic evaluation action-0 frequency",
        width=1450, height=700,
    )
    action0_figure.show()

## Curve-level comparison at a common budget

A single lucky ten-episode evaluation should not determine the ranking. The table therefore reports the time-average of the smoothed curve, its final ten-evaluation mean, its peak, its value at the common horizon, and first threshold crossings.

In [10]:
_trapezoid = getattr(np, "trapezoid", np.trapz)

def summarize_curve(frame):
    ordered = frame[frame["step"] <= comparison_budget].sort_values("step").copy()
    smoothed = ordered["raw_survival_pct"].rolling(SMOOTH_WINDOW, min_periods=1).mean()
    steps = ordered["step"].to_numpy(dtype=float)
    span = steps[-1] - steps[0] if len(steps) else 0
    result = {
        "evaluations": len(ordered),
        "curve_mean_pct": float(_trapezoid(smoothed.to_numpy(), steps) / span) if span > 0 else np.nan,
        "final_window_pct": float(smoothed.tail(FINAL_WINDOW_EVALS).mean()),
        "peak_pct": float(smoothed.max()),
        "at_common_budget_pct": float(smoothed.iloc[-1]),
    }
    for threshold in [60, 80, 90]:
        reached = steps[smoothed.to_numpy() >= threshold]
        result[f"steps_to_{threshold}pct_m"] = float(reached[0] / 1_000_000) if len(reached) else np.nan
    return result

curve_summary = pd.DataFrame([
    {"run_name": run_name, **summarize_curve(frame)}
    for run_name, frame in survival_long.groupby("run_name", sort=False)
])
actor_params = (
    history_df[history_df["metric"] == ACTOR_PARAM_METRIC]
    .groupby("run_name", as_index=False)["value"].max()
    .rename(columns={"value": "actor_params"})
)
curve_summary = (
    analysis_catalog.merge(curve_summary, on="run_name", how="inner")
    .merge(actor_params, on="run_name", how="left")
)
curve_ranking = curve_summary.sort_values("curve_mean_pct", ascending=False)
display(curve_ranking[[
    "variant", "seed", "actor_params", "curve_mean_pct",
    "final_window_pct", "peak_pct", "at_common_budget_pct",
    "steps_to_80pct_m", "steps_to_90pct_m",
]].round(2))

curve_rank_figure = px.bar(
    curve_ranking.sort_values("curve_mean_pct"),
    x="curve_mean_pct", y="variant", color="pool_label",
    orientation="h", hover_data=["seed", "final_window_pct", "peak_pct", "actor_params"],
    title=f"Curve mean up to {comparison_budget / 1_000_000:.3f}M steps",
    labels={"curve_mean_pct": "Time-averaged smoothed survival (%)", "variant": "Variant"},
    height=560,
)
curve_rank_figure.show()

,variant,seed,actor_params,curve_mean_pct,final_window_pct,peak_pct,at_common_budget_pct,steps_to_80pct_m,steps_to_90pct_m
0,mean_f0_a0h0,0,236550.0,59.31,87.03,93.30,86.17,4.73,4.98
3,mean_f1_a0h1,0,340617.0,53.82,88.87,95.09,92.75,4.64,5.31
5,tmean_f0_a0h1,0,483465.0,52.96,97.15,98.61,97.55,5.64,5.97
2,mean_f1_a0h0,0,241158.0,46.61,97.88,100.00,99.27,6.30,6.64
4,tmean_f0_a0h0,0,384006.0,46.37,63.80,69.28,62.58,NaN,NaN
6,tmean_f1_a0h0,0,388614.0,40.97,70.18,75.30,75.30,NaN,NaN
7,tmean_f1_a0h1,0,488073.0,36.57,82.28,92.58,92.58,7.63,7.88
1,mean_f0_a0h1,0,336009.0,29.88,39.17,45.42,44.60,NaN,NaN


## Paired marginal effects

Each difference matches the same seed and the same settings of the other two factors. Positive values favour the enabled or typed condition. With only seed 0 these are descriptive effects, not uncertainty estimates.

In [11]:
def paired_differences(frame, factor, low, high, controls, factor_label, metric):
    table = frame.pivot_table(index=["seed", *controls], columns=factor, values=metric, aggfunc="mean")
    if low not in table or high not in table:
        return pd.DataFrame()
    differences = (table[high] - table[low]).dropna().rename("difference_pp").reset_index()
    differences["factor"] = factor_label
    differences["metric"] = metric
    return differences

effect_frames = []
for metric in ["curve_mean_pct", "final_window_pct"]:
    effect_frames.extend([
        paired_differences(curve_summary, "pool", "mean", "typed_mean", ["use_action_features", "use_action0_head"], "typed mean − mean", metric),
        paired_differences(curve_summary, "use_action_features", False, True, ["pool", "use_action0_head"], "features on − off", metric),
        paired_differences(curve_summary, "use_action0_head", False, True, ["pool", "use_action_features"], "dedicated action-0 − shared", metric),
    ])
paired_effects = pd.concat([frame for frame in effect_frames if not frame.empty], ignore_index=True)
effect_summary = (
    paired_effects.groupby(["metric", "factor"], as_index=False)
    .agg(mean_difference_pp=("difference_pp", "mean"), comparisons=("difference_pp", "size"), min_difference_pp=("difference_pp", "min"), max_difference_pp=("difference_pp", "max"))
)
display(effect_summary.round(2))
effect_figure = px.bar(
    effect_summary, x="mean_difference_pp", y="factor", color="metric",
    barmode="group", orientation="h", hover_data=["comparisons", "min_difference_pp", "max_difference_pp"],
    title="Paired marginal effects across the other factor settings",
    labels={"mean_difference_pp": "Mean paired survival difference (percentage points)", "factor": "Contrast"},
    height=480,
)
effect_figure.add_vline(x=0, line_dash="dot", line_color="#6b7280")
effect_figure.show()

,metric,factor,mean_difference_pp,comparisons,min_difference_pp,max_difference_pp
0,curve_mean_pct,dedicated action-0 − shared,-5.01,4,-29.43,7.20
1,curve_mean_pct,features on − off,-2.64,4,-16.39,23.93
2,curve_mean_pct,typed mean − mean,-3.19,4,-17.25,23.08
3,final_window_pct,dedicated action-0 − shared,-2.86,4,-47.86,33.35
4,final_window_pct,features on − off,13.01,4,-14.87,49.69
5,final_window_pct,typed mean − mean,0.11,4,-27.70,57.97


## Full-test evaluation of the selected best checkpoints

This cell recursively scans `outputs/full_test_eval`. It accepts arbitrary subfolder and result filenames because the run identity is recovered from the checkpoint recorded inside each JSON. Only deterministic evaluations covering all 201 test chronics are marked valid for the headline comparison.

If multiple JSON files exist for a run, all duplicates are reported and the newest `created_at` result is retained.

In [12]:
def checkpoint_run_name(record):
    checkpoint_stem = Path(str(record.get("checkpoint", ""))).stem
    prefix = "best_test_"
    return checkpoint_stem[len(prefix):] if checkpoint_stem.startswith(prefix) else None

full_test_rows = []
for result_path in sorted(FULL_TEST_EVAL_DIR.rglob("*.json")):
    try:
        with result_path.open("r", encoding="utf-8") as file:
            record = json.load(file)
    except (OSError, json.JSONDecodeError):
        continue
    run_name = checkpoint_run_name(record)
    if run_name not in requested_run_name_set:
        continue
    full_test_rows.append({
        "run_name": run_name,
        "checkpoint_step": int(record.get("checkpoint_global_step", 0)),
        "survival_pct": float(record.get("survival_percent", np.nan)),
        "episodes": int(record.get("eval_episodes", 0)),
        "split": str(record.get("split", "")),
        "deterministic": bool(record.get("deterministic_eval", False)),
        "all_chronics": bool(record.get("eval_all_split_chronics", False)),
        "created_at": str(record.get("created_at", "")),
        "result_json": str(result_path.relative_to(wm.TASK_DIR)),
    })

full_test_results = pd.DataFrame(full_test_rows)
endpoint_df = pd.DataFrame()
if full_test_results.empty:
    print("No cas_hl full-test JSON files found yet. Re-run this cell after downloading them.")
else:
    duplicates = full_test_results[full_test_results.duplicated("run_name", keep=False)]
    if not duplicates.empty:
        print("Multiple evaluations found; retaining the newest created_at per run:")
        display(duplicates.sort_values(["run_name", "created_at"])[["run_name", "created_at", "result_json"]])
    full_test_results = (
        full_test_results.sort_values(["run_name", "created_at"])
        .drop_duplicates("run_name", keep="last")
    )
    full_test_results["valid_full_test"] = (
        full_test_results["split"].eq("test")
        & full_test_results["deterministic"]
        & full_test_results["all_chronics"]
        & full_test_results["episodes"].eq(EXPECTED_FULL_TEST_EPISODES)
    )
    invalid = full_test_results[~full_test_results["valid_full_test"]]
    if not invalid.empty:
        print("WARNING: excluded evaluations that are not deterministic 201-chronic test evaluations:")
        display(invalid[["run_name", "split", "deterministic", "all_chronics", "episodes", "result_json"]])
    valid_results = full_test_results[full_test_results["valid_full_test"]].copy()
    endpoint_df = analysis_catalog.merge(valid_results, on="run_name", how="inner")
    endpoint_df["checkpoint_step_m"] = endpoint_df["checkpoint_step"] / 1_000_000
    print(f"Valid full-test evaluations: {len(endpoint_df)} / {len(analysis_catalog)} downloaded runs")
    missing_evals = sorted(set(analysis_catalog["run_name"]) - set(endpoint_df["run_name"]))
    if missing_evals:
        print("Downloaded training runs still awaiting valid full-test evaluation:")
        for name in missing_evals:
            print("  ", name)
    if not endpoint_df.empty:
        display(endpoint_df.sort_values("survival_pct", ascending=False)[[
            "variant", "seed", "checkpoint_step_m", "survival_pct",
            "episodes", "result_json",
        ]].round(2))
        full_test_figure = px.bar(
            endpoint_df.sort_values("survival_pct"),
            x="survival_pct", y="variant", color="pool_label", orientation="h",
            hover_data=["seed", "checkpoint_step_m", "run_name"],
            title="Full-test survival of each selected best checkpoint",
            labels={"survival_pct": "Survival over all test chronics (%)", "variant": "Variant"},
            range_x=[0, 100], height=560,
        )
        full_test_figure.show()

No cas_hl full-test JSON files found yet. Re-run this cell after downloading them.


In [13]:
if endpoint_df.empty:
    print("Full-test matrices will appear here once valid evaluations are available.")
else:
    feature_labels = ["f0", "f1"]
    action0_labels = ["a0h0", "a0h1"]
    for pool in POOL_ORDER:
        subset = endpoint_df[endpoint_df["pool"] == pool]
        matrix = subset.pivot_table(
            index="action0_code", columns="feature_code", values="survival_pct", aggfunc="mean"
        ).reindex(index=action0_labels, columns=feature_labels)
        print(f"{POOL_LABELS[pool]} pooling — mean full-test survival (%)")
        display(matrix.round(2))
        if matrix.notna().any().any():
            figure = px.imshow(
                matrix, text_auto=".1f", aspect="auto", zmin=0, zmax=100,
                color_continuous_scale="Viridis",
                title=f"{POOL_LABELS[pool]} pooling — full-test survival",
                labels={"x": "Static action features", "y": "Action-0 head", "color": "Survival (%)"},
            )
            figure.show()

    comparison = endpoint_df[["run_name", "survival_pct"]].merge(
        curve_summary[["run_name", "variant", "curve_mean_pct", "final_window_pct"]],
        on="run_name", how="inner",
    )
    if not comparison.empty:
        comparison_figure = px.scatter(
            comparison, x="curve_mean_pct", y="survival_pct", text="variant",
            hover_data=["run_name", "final_window_pct"],
            title="Training-curve mean versus full-test best-checkpoint survival",
            labels={"curve_mean_pct": "Training-curve mean (%)", "survival_pct": "Full-test survival (%)"},
            width=950, height=650,
        )
        comparison_figure.update_traces(textposition="top center")
        comparison_figure.show()

Full-test matrices will appear here once valid evaluations are available.


## Compact result summary

The first line reports the most stable ranking available now. Once full-test results exist, the second line reports their winner separately rather than replacing the learning-dynamics result.

In [14]:
best_curve = curve_ranking.iloc[0]
print(
    f"Best curve mean: {best_curve['variant']} (seed {int(best_curve['seed'])}) — "
    f"{best_curve['curve_mean_pct']:.2f}% through {comparison_budget / 1_000_000:.3f}M steps; "
    f"final-window {best_curve['final_window_pct']:.2f}%."
)
if endpoint_df.empty:
    print("Full-test winner: pending.")
else:
    best_full_test = endpoint_df.sort_values("survival_pct", ascending=False).iloc[0]
    print(
        f"Best full-test checkpoint: {best_full_test['variant']} "
        f"(seed {int(best_full_test['seed'])}) — {best_full_test['survival_pct']:.2f}%."
    )

Best curve mean: mean_f0_a0h0 (seed 0) — 59.31% through 7.963M steps; final-window 87.03%.
Full-test winner: pending.


## Optional CSV export

In [15]:
EXPORT_TABLES = False
if EXPORT_TABLES:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    coverage.to_csv(EXPORT_DIR / "coverage.csv", index=False)
    curve_summary.to_csv(EXPORT_DIR / "curve_summary.csv", index=False)
    paired_effects.to_csv(EXPORT_DIR / "paired_effects.csv", index=False)
    effect_summary.to_csv(EXPORT_DIR / "effect_summary.csv", index=False)
    if not endpoint_df.empty:
        endpoint_df.to_csv(EXPORT_DIR / "full_test_results.csv", index=False)
    print("Saved tables under", EXPORT_DIR)
else:
    print("Set EXPORT_TABLES = True to write CSV files.")

Set EXPORT_TABLES = True to write CSV files.
